Please use T4 GPU

In [ ]:
!git clone https://github.com/deepseek-ai/DeepSeek-OCR-2.git
%cd DeepSeek-OCR-2
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install -r requirements.txt
!pip install flash-attn==2.7.3 --no-build-isolation

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
model_name = "deepseek-ai/DeepSeek-OCR-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)

model = model.eval()

In [ ]:
%cd ..

In [ ]:
import os

zip_file_path = "/content/data.zip"
output_dir = "documents"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Unzip the file
!unzip -o "{zip_file_path}" -d "{output_dir}"

print(f"Unzipped {zip_file_path} to {output_dir}")

In [ ]:
import os
import time
import psutil
import subprocess
import threading
import json
from datetime import datetime


# --- Global Config ---
INPUT_DIR = "/content/documents"
OUTPUT_DIR = "output"

CSV_LOG_NAME = "log.csv"
JSON_LOG_NAME = "log.json"

# --- GPU MONITOR ---
def get_gpu_stats():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8")

        util, mem_used, mem_total = result.strip().split(", ")
        return float(util), float(mem_used), float(mem_total)
    except:
        return 0.0, 0.0, 0.0


def get_gpu_temp():
    try:
        temp = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=temperature.gpu",
             "--format=csv,nounits,noheader"]
        ).decode("utf-8").strip()
        return float(temp)
    except:
        return 0.0


def batch_ocr(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    prompt = "<image>\n<|grounding|>Convert the image to markdown. "

    # --- LOG FILES ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_log_file = os.path.join(output_dir, CSV_LOG_NAME)
    json_log_file = os.path.join(output_dir, JSON_LOG_NAME)

    # CSV header
    with open(csv_log_file, "w") as f:
        f.write("filename,status,time_sec,cpu_percent,ram_peak_mb,"
                "gpu_util,gpu_mem,gpu_total,gpu_mem_percent,gpu_temp_peak\n")

    json_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "method": "DeepSeek-OCR-2"
        },
        "records": []
    }

    files = []

    for root, dirs, filenames in os.walk(INPUT_DIR):
        for f in filenames:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".pdf", ".tiff")):
                full_path = os.path.join(root, f)
                files.append(full_path)

    if not files:
        print("❌ No files found")
        return

    print("=" * 50)
    print("🚀 STARTING BATCH OCR")
    print("=" * 50)

    for i, filename in enumerate(files, 1):
        file_path = os.path.join(input_dir, filename)

        relative_path = os.path.relpath(file_path, input_dir)
        name_no_ext = os.path.splitext(relative_path)[0]

        output_path = os.path.join(output_dir, name_no_ext)
        os.makedirs(output_path, exist_ok=True)

        print(f"\n[{i}/{len(files)}] Processing: {filename}")

        # --- MONITORING ---
        cpu_log, ram_log, gpu_log, temp_log = [], [], [], []
        running = True

        start_time = time.time()
        current_pid = os.getpid()

        def monitor():
            while running:
                try:
                    # CPU
                    cpu_log.append(psutil.cpu_percent(interval=None))

                    # RAM (process-specific)
                    try:
                        ram = psutil.Process(current_pid).memory_info().rss / (1024 ** 2)
                        ram_log.append(ram)
                    except:
                        pass

                    # GPU
                    util, mem_used, mem_total = get_gpu_stats()
                    gpu_log.append((util, mem_used, mem_total))

                    # Temp
                    temp = get_gpu_temp()
                    if temp:
                        temp_log.append(temp)

                    time.sleep(1)

                except Exception as e:
                    print(f"Monitor error: {e}")
                    break

        thread = threading.Thread(target=monitor, daemon=True)
        thread.start()

        # --- RUN OCR ---
        try:
            res = model.infer(
                tokenizer,
                prompt=prompt,
                image_file=file_path,
                output_path=output_path,
                base_size=1024,
                image_size=640,
                crop_mode=True,
                save_results=True,
                test_compress=True
            )

            status = "SUCCESS"

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            status = "ERROR"

        # --- STOP MONITOR ---
        running = False
        thread.join(timeout=2)

        elapsed = time.time() - start_time

        # --- METRICS ---
        cpu_avg = sum(cpu_log) / len(cpu_log) if cpu_log else 0
        ram_peak = max(ram_log) if ram_log else 0

        if gpu_log:
            gpu_util = sum(x[0] for x in gpu_log) / len(gpu_log)
            gpu_mem = sum(x[1] for x in gpu_log) / len(gpu_log)
            gpu_total = gpu_log[0][2]
            gpu_mem_percent = (gpu_mem / gpu_total * 100) if gpu_total else 0
        else:
            gpu_util = gpu_mem = gpu_total = gpu_mem_percent = 0

        temp_peak = max(temp_log) if temp_log else 0

        # --- PRINT ---
        print(f"✅ {status} | {elapsed:.2f}s")
        print(f"CPU: {cpu_avg:.1f}% | RAM peak: {ram_peak:.0f}MB")
        print(f"GPU: {gpu_util:.1f}% | VRAM: {gpu_mem:.0f}/{gpu_total:.0f}MB ({gpu_mem_percent:.1f}%)")
        if temp_peak:
            print(f"GPU Temp: {temp_peak:.1f}°C")

        # --- CSV LOG ---
        with open(csv_log_file, "a") as f:
            f.write(f"{filename},{status},{elapsed:.2f},{cpu_avg:.1f},{ram_peak:.0f},"
                    f"{gpu_util:.1f},{gpu_mem:.0f},{gpu_total:.0f},{gpu_mem_percent:.1f},{temp_peak:.1f}\n")

        # --- JSON LOG ---
        json_data["records"].append({
            "filename": filename,
            "status": status,
            "time_sec": round(elapsed, 2),
            "cpu_percent": round(cpu_avg, 1),
            "ram_peak_mb": round(ram_peak, 0),
            "gpu": {
                "util": round(gpu_util, 1),
                "mem_used": round(gpu_mem, 0),
                "mem_total": round(gpu_total, 0),
                "mem_percent": round(gpu_mem_percent, 1),
                "temp_peak": round(temp_peak, 1)
            }
        })

    # Save JSON
    with open(json_log_file, "w") as f:
        json.dump(json_data, f, indent=2)

    print("\n🎉 DONE")
    print(f"CSV: {csv_log_file}")
    print(f"JSON: {json_log_file}")

In [ ]:
import json
import os

# Function to calculate overall summary for the processing time, resources utilization
def calculate_summary(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    records = data.get("records", [])
    if not records:
        print("No records found.")
        return

    total_files = len(records)

    # Accumulators
    total_time = 0
    total_cpu = 0
    total_ram = 0
    total_gpu_util = 0
    total_gpu_mem = 0
    total_gpu_mem_percent = 0
    total_gpu_temp = 0

    for r in records:
        total_time += r["time_sec"]
        total_cpu += r["cpu_percent"]
        total_ram += r["ram_peak_mb"]

        gpu = r.get("gpu", {})
        total_gpu_util += gpu.get("util", 0)
        total_gpu_mem += gpu.get("mem_used", 0)
        total_gpu_mem_percent += gpu.get("mem_percent", 0)
        total_gpu_temp += gpu.get("temp_peak", 0)

    # Averages
    summary = {
        "total_files": total_files,
        "avg_time_sec": total_time / total_files,
        "avg_cpu_percent": total_cpu / total_files,
        "avg_ram_peak_mb": total_ram / total_files,
        "avg_gpu_util": total_gpu_util / total_files,
        "avg_gpu_mem_mb": total_gpu_mem / total_files,
        "avg_gpu_mem_percent": total_gpu_mem_percent / total_files,
        "avg_gpu_temp": total_gpu_temp / total_files
    }

    # ---- SAVE TO TXT ----
    txt_file = os.path.splitext(json_file)[0] + "_summary.txt"

    with open(txt_file, "w") as f:
        f.write("===== OVERALL SUMMARY =====\n")
        for k, v in summary.items():
            if isinstance(v, float):
                f.write(f"{k}: {v:.2f}\n")
            else:
                f.write(f"{k}: {v}\n")

    print(f"✅ Summary saved to: {txt_file}")

    return summary


In [ ]:
batch_ocr(INPUT_DIR, OUTPUT_DIR)

JSON_FILE = os.path.join(OUTPUT_DIR, JSON_LOG_NAME)
calculate_summary(JSON_FILE)

In [ ]:
!cd /content && zip -r deepseek-OCR2_output.zip $OUTPUT_DIR

from google.colab import files
files.download("/content/deepseek-OCR2_output.zip")